In [6]:
from sklearn.linear_model import LinearRegression

In [7]:
model = LinearRegression()

In [9]:
type(model).__name__

'LinearRegression'

In [4]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 1. Create a sample dataset (independent variables only)
data = {
    'Age': [25, 30, 35, 40, 45, 50],
    'Salary': [50000, 60000, 80000, 85000, 110000, 115000],
    'Years_Experience': [2, 5, 7, 9, 12, 14],
    'Savings': [10000, 15000, 30000, 25000, 60000, 55000]
}
df = pd.DataFrame(data)

# 2. Crucial Step: Add a constant (intercept) column if it's not already there.
# Statsmodels' VIF function requires an explicit intercept column to calculate correctly.
df_with_const = df.copy()
#df_with_const['intercept'] = 1

# 3. Calculate VIF for each column
vif_data = pd.DataFrame()
vif_data["Feature"] = df.columns

# Calculate VIF only for the original features (excluding the intercept row from the output)
vif_data["VIF"] = [
    variance_inflation_factor(df_with_const.values, i) 
    for i in range(df.shape[1])
]

print(vif_data)


            Feature          VIF
0               Age  3011.013082
1            Salary  4454.312693
2  Years_Experience    60.656775
3           Savings   244.515442


In [16]:
import numpy as np
import pandas as pd

# Sample 2D array (4 rows, 3 columns)
arr = np.array([
    [ 1.5, -2.0,  0.0, 1],
    [ 3.2,  1.1, -1.2, 2],
    [ 0.0, -4.5, -3.3, 3],
    [ 2.1,  0.0, -0.5, 4]
])

# 1. Get the signs
signs = np.sign(arr)

# 2. Count occurrences along axis=0 (down each column)
pos_counts = (signs == 1).sum(axis=0)
neg_counts = (signs == -1).sum(axis=0)
zero_counts = (signs == 0).sum(axis=0)


In [17]:
pos_counts

array([3, 1, 0, 4])

In [18]:
counts = np.array(((signs==1).sum(axis=0), (signs==-1).sum(axis=0), (signs==0).sum(axis=0)))

In [19]:
counts / 4

array([[0.75, 0.25, 0.  , 1.  ],
       [0.  , 0.5 , 0.75, 0.  ],
       [0.25, 0.25, 0.25, 0.  ]])

In [20]:
np.max(counts / 4, axis = 0)

array([0.75, 0.5 , 0.75, 1.  ])

In [21]:
import tabulate

In [31]:
import pandas as pd
from tabulate import tabulate

# 1. Base Learners & Meta Learner Names
learners = ['RandomForest_Reg', 'GradientBoosting_Reg', 'XGBoost_Reg', 'Ridge_Baseline']
model_rows = learners + ['META_LEARNER (Ensemble)']

# 2. Re-structured Metrics Table (Base Learners vs Meta Learner)
# This lets you compare instantly how much performance the meta-learner gained
mock_metrics_df = pd.DataFrame(
    data=[
        [8.9412, 112.4501, 10.6042, 0.8214],  # RandomForest
        [8.4125,  99.8514,  9.9926, 0.8415],  # GradientBoosting
        [8.2251,  97.1042,  9.8541, 0.8502],  # XGBoost
        [9.8451, 142.1152, 11.9212, 0.7710],  # Ridge Baseline
        [6.8142,  68.2140,  8.2592, 0.8984]   # META LEARNER (Improved Performance)
    ],
    index=model_rows,
    columns=['MAE', 'MSE', 'RMSE', 'R² Score']
)

# 3. Correlation Matrix (Symmetric Dataframe - Base Learners Only)
mock_cor = pd.DataFrame(
    data=[
        [1.0000, 0.8842, 0.9125, 0.7631],
        [0.8842, 1.0000, 0.9314, 0.7412],
        [0.9125, 0.9314, 1.0000, 0.7105],
        [0.7631, 0.7412, 0.7105, 1.0000]
    ],
    index=learners,
    columns=learners
)

# 4. Variance Inflation Factor (VIF - Base Learners Only)
mock_vif = pd.DataFrame({
    'Feature': learners,
    'VIF': [12.45, 15.12, 18.64, 2.85]
})

# 5. Bootstrap CI & Sign Stability Table (Meta-Model Weights)
row_names = ['Intercept'] + learners
stats_names = ['mean', 'median', '2.5% quantile', '97.5% quantile', 'sign stability']

mock_ci = pd.DataFrame(
    data=[
        [ 2.1402,  2.1150,  0.8540,  3.4210, 1.0000],  # Intercept
        [ 0.3842,  0.3910, -0.0410,  0.7812, 0.9150],  # RF
        [ 0.1245,  0.1180, -0.2105,  0.4850, 0.7420],  # GB
        [ 0.4951,  0.5102,  0.1542,  0.8124, 1.0000],  # XGB
        [ 0.0714,  0.0695,  0.0125,  0.1412, 0.9950]   # Ridge
    ],
    index=row_names,
    columns=stats_names
)

summary_dict = {'metrics': mock_metrics_df, 'CI': mock_ci, 'cor': mock_cor, 'vif': mock_vif}


In [33]:
from tabulate import tabulate

def print_summary_report(summary_dict):
    print("=" * 70)
    print("                ENSEMBLE MODEL DIAGNOSTICS REPORT                ")
    print("=" * 70)
    
    # 1. Print Basic Metrics
    print("\n[BASIC PERFORMANCE METRICS]")
    print(tabulate(summary_dict['metrics'], headers ="keys", tablefmt="fancy_grid"))
    print("=" * 70)

    # 3. Print Bootstrap Confidence Intervals & Stability
    print("\n[BOOTSTRAP COEFFICIENT ANALYSIS (CI & SIGN STABILITY)]")
    print(tabulate(summary_dict['CI'], headers="keys", tablefmt="fancy_grid"))
    print("=" * 70)

    print("Correlation Matrix")
    print(tabulate(summary_dict['cor'], headers="keys", tablefmt="fancy_grid"))
    print("=" * 70)
        
    # 2. Print VIF Data
    print("\n[MULTICOLLINEARITY (VIF)]")
    # headers="keys" automatically uses DataFrame column titles
    print(tabulate(summary_dict['vif'], headers="keys", tablefmt="fancy_grid", showindex=False))
    

# Execute it with your summary dictionary
print_summary_report(summary_dict)


                ENSEMBLE MODEL DIAGNOSTICS REPORT                

[BASIC PERFORMANCE METRICS]
╒═════════════════════════╤════════╤══════════╤═════════╤════════════╕
│                         │    MAE │      MSE │    RMSE │   R² Score │
╞═════════════════════════╪════════╪══════════╪═════════╪════════════╡
│ RandomForest_Reg        │ 8.9412 │ 112.45   │ 10.6042 │     0.8214 │
├─────────────────────────┼────────┼──────────┼─────────┼────────────┤
│ GradientBoosting_Reg    │ 8.4125 │  99.8514 │  9.9926 │     0.8415 │
├─────────────────────────┼────────┼──────────┼─────────┼────────────┤
│ XGBoost_Reg             │ 8.2251 │  97.1042 │  9.8541 │     0.8502 │
├─────────────────────────┼────────┼──────────┼─────────┼────────────┤
│ Ridge_Baseline          │ 9.8451 │ 142.115  │ 11.9212 │     0.771  │
├─────────────────────────┼────────┼──────────┼─────────┼────────────┤
│ META_LEARNER (Ensemble) │ 6.8142 │  68.214  │  8.2592 │     0.8984 │
╘═════════════════════════╧════════╧══════════╧══════

In [34]:
arr = np.array([[1, 2],[3, 4]])

In [35]:
arr

array([[1, 2],
       [3, 4]])

In [37]:
df = pd.DataFrame(arr, columns = ['intercept', 'b1'])

In [38]:
df

,intercept,b1
0,1,2
1,3,4


In [39]:
cor = df.corr()

In [40]:
cor

,intercept,b1
intercept,1.0,1.0
b1,1.0,1.0


In [41]:
summary_dict

{'metrics':                             MAE       MSE     RMSE  R² Score
 RandomForest_Reg         8.9412  112.4501  10.6042    0.8214
 GradientBoosting_Reg     8.4125   99.8514   9.9926    0.8415
 XGBoost_Reg              8.2251   97.1042   9.8541    0.8502
 Ridge_Baseline           9.8451  142.1152  11.9212    0.7710
 META_LEARNER (Ensemble)  6.8142   68.2140   8.2592    0.8984,
 'CI':                         mean  median  2.5% quantile  97.5% quantile  \
 Intercept             2.1402  2.1150         0.8540          3.4210   
 RandomForest_Reg      0.3842  0.3910        -0.0410          0.7812   
 GradientBoosting_Reg  0.1245  0.1180        -0.2105          0.4850   
 XGBoost_Reg           0.4951  0.5102         0.1542          0.8124   
 Ridge_Baseline        0.0714  0.0695         0.0125          0.1412   
 
                       sign stability  
 Intercept                      1.000  
 RandomForest_Reg               0.915  
 GradientBoosting_Reg           0.742  
 XGBoost_Reg    